# Jarvis — Kaggle T4 ×2: ONE Ordered Notebook

**Based on your uploaded `jarvis-full-duplex.ipynb` and its executed outputs.** Your notebook previously detected two Tesla T4 GPUs, mounted *raw* English/Tamil reference files (not `voices.zip`), and generated OmniVoice audio in both languages. The previous full server later encountered CUDA out-of-memory and invalid-device errors.

**Before clicking Run All:**
1. **Stop Jarvis on Windows** using Ctrl+C.
2. **Restart the Kaggle session** to release old CUDA allocations and old server/tunnel processes. If Kaggle just required a kernel restart after installing packages, perform that restart once; **do not reinstall packages again when imports already work**.
3. In Kaggle **Input**, attach your existing **private** `jarvis-voice-references` dataset. Choose **GPU T4 ×2**, enable Internet (if offered).
4. **ROTATE the old bearer token** because it appeared in the uploaded notebook output. Add the new token as Kaggle secret named **`VOICE_API_TOKEN`** (Add-ons → Secrets). Save the **same value privately** for your Windows `.env`. **Never print the token.**

Run every code cell **from top to bottom**. If a cell fails, fix that cell before continuing. No separate TTS model is loaded in the notebook: the server does the Qwen and OmniVoice tests in its **own process**, avoiding duplicate GPU allocations.

## 1. Update project code (works for both an existing repo and a fresh notebook)

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

REPO = Path("/kaggle/working/jarvis-realtime-voice")
GIT_URL = "https://github.com/Godf-Cyb3r-1/jarvis-realtime-voice.git"

if (REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError("The project folder exists without .git. Rename/remove it manually, then rerun.")
    subprocess.run(["git", "clone", GIT_URL, str(REPO)], check=True)

os.chdir(REPO)
for name in ["duplex_server.py", "duplex_client.py", "gpu_guard.py",
             "tts_engine.py", "local_llm.py", "requirements-duplex-server.txt"]:
    assert (REPO / name).is_file(), f"Missing source file: {name}"
print("PASS: updated Jarvis repo at", REPO)

## 2. Check installed packages — only install if something is missing

Your uploaded notebook already successfully imported OmniVoice, Whisper, ONNX, and Transformers after installation. This cell **skips pip** when they are present. If it installs anything, it intentionally stops; **restart the Python session and rerun from Step 1** so CUDA and PyTorch see a consistent environment.

In [ ]:
import importlib.util
needed = {
    "torch": "torch", "ctranslate2": "ctranslate2", "faster_whisper": "faster_whisper",
    "omnivoice": "omnivoice", "transformers": "transformers",
    "fastapi": "fastapi", "uvicorn": "uvicorn", "soundfile": "soundfile",
    "httpx": "httpx", "scipy": "scipy", "requests": "requests",
    "onnxruntime": "onnxruntime"
}
missing = [name for name, mod in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Missing packages:", missing)
    if any(name != "omnivoice" for name in missing):
        subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                        str(REPO / "requirements-duplex-server.txt")], check=True)
    if "omnivoice" in missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "omnivoice"], check=True)
    raise RuntimeError("Packages installed. RESTART the Python session, then run this notebook again from Step 1.")
print("PASS: all required dependencies already installed; no reinstall was done")

## 3. Confirm BOTH frameworks can see GPU 0 and GPU 1

In [ ]:
import torch, ctranslate2, transformers, omnivoice, faster_whisper
print("PyTorch:", torch.__version__, "| Transformers:", transformers.__version__)
print("Torch CUDA GPUs:", torch.cuda.device_count())
print("CTranslate2 CUDA GPUs:", ctranslate2.get_cuda_device_count())
assert torch.cuda.is_available(), "Enable Kaggle GPU accelerator"
assert torch.cuda.device_count() >= 2, "This notebook requires T4 ×2; don't assign GPU 1 on a single GPU"
assert ctranslate2.get_cuda_device_count() >= 2, "CTranslate2 cannot see GPU 1; stop before loading models"
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {free / 2**30:.2f} / {total / 2**30:.2f} GiB free")
print("PASS: dual-GPU device assignment is available")

## 4. Prepare your ALREADY-ATTACHED private voices (no ZIP needed)

This uses the raw `jarvis_en/new-ref.mp3` and `jarvis_ta/ref.mp3` paths that were present in your uploaded notebook. The reference transcripts are copied unmodified.

In [ ]:
import shutil
import soundfile as sf

dataset_candidates = []
for en_file in Path("/kaggle/input").rglob("new-ref.mp3"):
    dataset = en_file.parent.parent
    required = [
        dataset / "jarvis_en/new-ref.mp3", dataset / "jarvis_en/ref.txt",
        dataset / "jarvis_ta/ref.mp3", dataset / "jarvis_ta/ref.txt"
    ]
    if en_file.parent.name == "jarvis_en" and all(p.is_file() for p in required):
        dataset_candidates.append(dataset)
assert dataset_candidates, ("Reference dataset not attached. Check Kaggle Input for your private "
                            "jarvis-voice-references dataset.")
DATASET = dataset_candidates[0]
VOICE_DIR = REPO / "voices"
VOICE_DIR.mkdir(parents=True, exist_ok=True)
for lang, audio_rel, text_rel in [
    ("en", "jarvis_en/new-ref.mp3", "jarvis_en/ref.txt"),
    ("ta", "jarvis_ta/ref.mp3", "jarvis_ta/ref.txt")
]:
    source = VOICE_DIR / f"{lang}.mp3"
    shutil.copy2(DATASET / audio_rel, source)
    shutil.copy2(DATASET / text_rel, VOICE_DIR / f"{lang}.txt")
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(source),
                    "-ac", "1", "-ar", "24000", str(VOICE_DIR / f"{lang}.wav")], check=True)
    info = sf.info(VOICE_DIR / f"{lang}.wav")
    assert info.samplerate == 24000 and info.frames > 0
    assert (VOICE_DIR / f"{lang}.txt").read_text(encoding="utf-8").strip()
    print(lang.upper(), "voice OK:", round(info.duration, 1), "seconds @", info.samplerate, "Hz")
print("PASS: your English and Tamil voice references are ready (still private)")

## 5. Secure token, GPU assignments, CUDA runtime paths — **before** server start

Initial settings: Whisper Small + small Qwen on **GPU 0**, OmniVoice on **GPU 1**. Interim partial ASR and Smart Turn are temporarily disabled while we prove the complete speaking pipeline. Continuous microphone capture is still enabled. Once speaking works, re-enable extra features one at a time.

In [ ]:
import site
from kaggle_secrets import UserSecretsClient
try:
    secret = UserSecretsClient().get_secret("VOICE_API_TOKEN")
except Exception as exc:
    raise RuntimeError("Create/enable Kaggle Secret VOICE_API_TOKEN first. Do not print it.") from exc
assert isinstance(secret, str) and len(secret) >= 24, "Use a NEW random token at least 24 characters long."
os.environ["VOICE_API_TOKEN"] = secret
del secret

os.environ.update({
    "ASR_DEVICE": "cuda",
    "ASR_GPU_INDEX": "0",
    "ASR_MODEL": "small",
    "ASR_COMPUTE": "int8_float16",
    "ASR_BEAM_SIZE": "1",
    "ASR_LANGUAGE": "auto",
    "PARTIAL_ASR_ENABLED": "0",
    "SMART_TURN_ENABLED": "0",
    "LLM_BACKEND": "local",
    "LOCAL_LLM_MODEL": "Qwen/Qwen3-0.6B",
    "LOCAL_LLM_DEVICE": "cuda:0",
    "LOCAL_LLM_GPU_INDEX": "0",
    "LOCAL_LLM_MAX_TOKENS": "100",
    "TTS_PROVIDER": "omnivoice",
    "TTS_GPU": "1",
    "VOICE_DIR": str(VOICE_DIR),
    "OMNIVOICE_STEPS": "12",
    "ALLOW_BACKCHANNEL": "0",
    "PYTHONUNBUFFERED": "1",
})
# The previous notebook updated CUDA paths AFTER the server launched.
# Here the NEW server process inherits them before importing CUDA libraries.
cuda_libs = []
for site_dir in site.getsitepackages():
    for subdir in ("cublas", "cudnn", "cuda_runtime"):
        candidate = Path(site_dir) / "nvidia" / subdir / "lib"
        if candidate.is_dir():
            cuda_libs.append(str(candidate))
existing = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = ":".join(dict.fromkeys(
    cuda_libs + ([existing] if existing else [])
))
print("PASS: environment configured; secret loaded but NOT printed")
print("GPU layout: Whisper+Qwen = GPU 0 | OmniVoice = GPU 1")
print("CUDA library directories found:", len(cuda_libs))

## 6. Check the actual source and unit tests

In [ ]:
subprocess.run([sys.executable, "-m", "compileall", "-q",
    "gpu_guard.py", "duplex_server.py", "duplex_client.py",
    "tts_engine.py", "local_llm.py", "smart_turn_adapter.py"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover",
    "-s", "tests", "-p", "test_*.py", "-v"], check=True)
print("PASS: syntax and policy tests")

## 7. Start ONE server only

Do **not** run `OmniVoiceEngine()` directly in another Kaggle cell: that can load a duplicate GPU model. This cell checks whether an older process still owns port 8091 and refuses to silently use the wrong server.

In [ ]:
import socket
import requests

if "server" in globals() and server is not None and server.poll() is None:
    server.terminate()
    server.wait(timeout=20)
    time.sleep(2)

with socket.socket() as sock:
    port_busy = sock.connect_ex(("127.0.0.1", 8091)) == 0
if port_busy:
    raise RuntimeError("Port 8091 is already in use by an OLD server. Stop it or restart Kaggle session.")

SERVER_LOG = Path("/kaggle/working/jarvis_server.log")
server_log_handle = SERVER_LOG.open("w")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "duplex_server:app",
     "--host", "127.0.0.1", "--port", "8091"],
    cwd=REPO, env=os.environ.copy(),
    stdout=server_log_handle, stderr=subprocess.STDOUT
)
health = None
for _ in range(180):
    if server.poll() is not None:
        break
    try:
        response = requests.get("http://127.0.0.1:8091/health", timeout=2)
        if response.ok:
            health = response.json()
            break
    except requests.RequestException:
        pass
    time.sleep(1)
if not health:
    print(SERVER_LOG.read_text(errors="replace")[-5000:])
    raise RuntimeError("Jarvis server did not start. See log above.")

print("SERVER:", {k: health.get(k) for k in
    ("ready", "asr_model", "asr_device", "asr_gpu_index", "llm_backend",
     "tts_provider", "smart_turn")})
assert health.get("asr_gpu_index") == 0, "Server did not use GPU 0 for Whisper"
assert health.get("tts_provider") == "omnivoice"
print("PASS: one correct Jarvis server is running")

## 8. Test Qwen inside the running server (no duplicate model)

The first call may download model weights. **Stop if this fails; don't start the tunnel yet.**

In [ ]:
HEADERS = {"Authorization": "Bearer " + os.environ["VOICE_API_TOKEN"]}
try:
    r = requests.post(
        "http://127.0.0.1:8091/debug/llm", headers=HEADERS,
        json={"text": "Hello Jarvis. Please reply with a friendly, short sentence."},
        timeout=300
    )
    print("LLM HTTP:", r.status_code)
    print("LLM response:", r.text[:700])
    if not r.ok or not r.json().get("reply", "").strip():
        print("SERVER LOG:", SERVER_LOG.read_text(errors="replace")[-2500:])
        raise RuntimeError("Qwen did not generate a reply; fix GPU0/LLM before continuing.")
except requests.RequestException:
    print("SERVER LOG:", SERVER_LOG.read_text(errors="replace")[-3000:])
    raise
print("PASS: reasoning works")

## 9. Test OmniVoice English AND Tamil inside the SAME server

This proves real server speech generation, not merely a separate notebook demo. Audio is delivered as WAV with the **correct embedded sample rate** (your old Tamil test played its 16-kHz audio at 14.2 kHz by mistake).

In [ ]:
from IPython.display import Audio, display
for lang, text in [
    ("en", "Hey, I'm Jarvis. Nice to hear from you."),
    ("ta", "வணக்கம்! நான் ஜார்விஸ். எப்படி இருக்கீங்க?")
]:
    try:
        r = requests.post("http://127.0.0.1:8091/debug/tts",
            headers=HEADERS, json={"text": text, "language": lang}, timeout=300)
    except requests.RequestException:
        print("SERVER LOG:", SERVER_LOG.read_text(errors="replace")[-3000:])
        raise
    print(f"OmniVoice {lang} HTTP:", r.status_code)
    if not r.ok:
        print("Response:", r.text[:1000])
        print("SERVER LOG:", SERVER_LOG.read_text(errors="replace")[-3000:])
        raise RuntimeError(f"OmniVoice {lang} failed; do not continue yet.")
    target = Path("/kaggle/working") / f"jarvis_checked_{lang}.wav"
    target.write_bytes(r.content)
    info = sf.info(target)
    assert info.frames > 0 and info.samplerate == 16000
    print("Voice audio:", round(info.duration, 2), "seconds, 16000 Hz")
    display(Audio(filename=str(target)))
print("PASS: both English and Tamil speech synthesis work in the SERVER")

## 10. Create ONE temporary authenticated WebSocket tunnel

Use this only after **both** debug tests pass. Cloudflare Quick Tunnel URLs are temporary and publicly reachable; the server separately checks the private Authorization token. Do not share your secret.

In [ ]:
import re
CLOUDFLARED = Path("/kaggle/working/cloudflared")
if not CLOUDFLARED.is_file() or CLOUDFLARED.stat().st_size < 100000:
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", str(CLOUDFLARED)], check=True)
CLOUDFLARED.chmod(0o755)

if "tunnel" in globals() and tunnel is not None and tunnel.poll() is None:
    tunnel.terminate()
    tunnel.wait(timeout=15)

TUNNEL_LOG = Path("/kaggle/working/cloudflared.log")
tunnel_log_handle = TUNNEL_LOG.open("w")
tunnel = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--url", "http://127.0.0.1:8091"],
    stdout=tunnel_log_handle, stderr=subprocess.STDOUT
)
WS_URL = None
for _ in range(60):
    if tunnel.poll() is not None:
        break
    time.sleep(1)
    matches = re.findall(
        r"https://[a-z0-9-]+\.trycloudflare\.com",
        TUNNEL_LOG.read_text(errors="replace")
    )
    if matches:
        WS_URL = matches[-1].replace("https://", "wss://") + "/ws"
        break
if not WS_URL:
    print(TUNNEL_LOG.read_text(errors="replace")[-2000:])
    raise RuntimeError("Tunnel did not start. Check Kaggle Internet permissions.")
print("VOICE_WS_URL =", WS_URL)
print("PASS: use this URL in Windows .env and the SAME rotated token from Kaggle Secrets")

## 11. Windows PowerShell — run on your PC, NOT in Kaggle

Use headphones during the first test. In **PowerShell**:

```powershell
cd C:\Users\LENOVO\jarvis-realtime-voice
git pull --ff-only
.\.venv\Scripts\python.exe -m pip install -r requirements-duplex-client.txt

# First verify your Windows mic and speaker work simultaneously:
.\.venv\Scripts\python.exe local_audio_smoke.py

# Update only connection settings:
notepad .env
```

Edit your existing Windows `.env` (do not put this in GitHub):

```ini
VOICE_WS_URL=wss://THE-URL-FROM-STEP-10/ws
VOICE_API_TOKEN=YOUR-NEW-KAGGLE-SECRET-VALUE
AEC_ENABLED=0
VAD_MODE=2
```

Then start:

```powershell
.\.venv\Scripts\python.exe duplex_client.py
```

Speak **once**, then wait. Expect `[transcript]`, `[stage:llm]`, streamed reply text, `[stage:tts]`, and `[metric] first_pcm_after_commit_ms`, followed by audible speech. **The initial configuration intentionally disables partial-ASR; that is normal.**

If Kaggle restarted after Step 10, the tunnel URL may have changed. Update `.env` accordingly.

## 12. Read server logs if anything fails (optional)

This cell does not restart anything. Only run it if your Windows output has an error or silence.

In [ ]:
print("Server running:", server.poll() is None)
print("Tunnel running:", "tunnel" in globals() and tunnel.poll() is None)
print("Server log (last 4500 characters):")
print(SERVER_LOG.read_text(errors="replace")[-4500:])

## 13. After FIRST spoken reply: enable richer duplex incrementally

Only **after** the server's Qwen/OmniVoice diagnostics pass AND Windows plays Jarvis speech:

1. Stop Windows client with **Ctrl+C**.
2. In the Kaggle Step 5 settings, change `PARTIAL_ASR_ENABLED` from `"0"` to `"1"` and run **Step 5 → Step 7** (server restart). The existing tunnel may stay running. Reconnect Windows; verify again.
3. Then try `SMART_TURN_ENABLED="1"` and restart the server again. Its Tamil/Tanglish accuracy needs measurement. If model loading fails, revert to `"0"`.
4. Test a natural pause and a deliberate interruption with headphones.

**What this notebook solves:** a correct install/restart order, raw dataset paths, no exposed tokens, GPU0/GPU1 assignments, smaller memory footprint, isolated in-server Qwen and OmniVoice tests, one server, one tunnel, Windows audio check. It **does not by itself prove** native human-level full-duplex conversation or solve every hardware/latency issue.